In [1]:
import numpy as np
import pandas as pd

In [2]:
np.random.seed(42)

N = 50000 # 50K rows

Flight Info

In [3]:
flight_id = np.arange(1, N+1)
aircraft_id = np.random.choice([f"AC_{i}" for i in range(1, 201)], N)
flight_hours = np.random.randint(0, 601, N)  # 0–600 hrs since last maintenance
flight_cycles = np.random.randint(0, 301, N) # 0–300 cycles since last maintenance

Sensor Data

In [4]:
engine_temp = np.random.normal(850, 50, N)  # Normal ~850°C, SD=50, some extremes
vibration_level = np.abs(np.random.normal(1.5, 0.7, N))  # g-force
oil_pressure = np.random.normal(50, 10, N)  # psi, normal ~50
fuel_flow = np.random.normal(5000, 1200, N)  # kg/hr
cabin_pressure = np.random.normal(12, 1, N)  # psi

Environment/Route

In [5]:
route_type = np.random.choice(["Domestic", "International", "Long-haul"], N, p=[0.5, 0.3, 0.2])
avg_weather_temp = np.random.uniform(-30, 45, N)  # °C
turbulence = np.random.choice([0,1,2,3,4,5], N, p=[0.25,0.25,0.2,0.15,0.1,0.05])
airport_condition = np.round(np.random.uniform(0.0, 1.0, N), 2)  # 0=bad, 1=perfect

Maintenance History

In [6]:
# --- Maintenance History ---
last_maintenance_days = np.random.randint(1, 181, N)  # 1–180 days
maintenance_score = np.round(np.random.uniform(0.4, 1.0, N), 2)

Failure Probability Logic

In [7]:
# Base probability ~2%
prob_failure = 0.02 + \
               0.02*(engine_temp > 950) + \
               0.03*(vibration_level > 3) + \
               0.03*(oil_pressure < 25) + \
               0.02*(flight_hours > 500) + \
               0.02*(flight_cycles > 250) + \
               0.03*(maintenance_score < 0.7) + \
               0.02*(turbulence > 3) + \
               0.02*( (avg_weather_temp < -20) | (avg_weather_temp > 40) ) + \
               0.02*(airport_condition < 0.3)


prob_failure = np.clip(prob_failure, 0, 0.9)  # Clip probabilities between 0 and 0.9


failure_within_30_days = np.random.binomial(1, prob_failure) # Sample failures from Bernoulli distribution

Create dataframe

In [8]:
data = pd.DataFrame({
    "flight_id": flight_id,
    "aircraft_id": aircraft_id,
    "flight_hours": flight_hours,
    "flight_cycles": flight_cycles,
    "engine_temp": engine_temp.round(2),
    "vibration_level": vibration_level.round(2),
    "oil_pressure": oil_pressure.round(2),
    "fuel_flow": fuel_flow.round(2),
    "cabin_pressure": cabin_pressure.round(2),
    "route_type": route_type,
    "avg_weather_temp": avg_weather_temp.round(2),
    "turbulence": turbulence,
    "airport_condition": airport_condition,
    "last_maintenance_days": last_maintenance_days,
    "maintenance_score": maintenance_score,
    "failure_within_30_days": failure_within_30_days
})

Save to CSV

In [11]:
data.to_csv("/content/drive/MyDrive/Case Studies/United Airlines/dataset/airline_maintenance.csv", index=False)

In [9]:
print(data.head())
print("\nDataset shape:", data.shape)
print("Failure rate:", data['failure_within_30_days'].mean())

   flight_id aircraft_id  flight_hours  flight_cycles  engine_temp  \
0          1      AC_103           307            100       863.51   
1          2      AC_180           371             78       765.82   
2          3       AC_93           214            270       894.46   
3          4       AC_15           261            183       860.48   
4          5      AC_107           345             81       935.12   

   vibration_level  oil_pressure  fuel_flow  cabin_pressure route_type  \
0             1.23         43.10    6074.29           12.67   Domestic   
1             1.86         54.47    6423.24           12.30  Long-haul   
2             2.00         60.74    5694.91           11.35   Domestic   
3             1.45         52.37    6198.98           10.16   Domestic   
4             1.77         55.67    3752.87           12.65   Domestic   

   avg_weather_temp  turbulence  airport_condition  last_maintenance_days  \
0             17.74           0               0.64       

In [10]:
data

,flight_id,aircraft_id,flight_hours,flight_cycles,engine_temp,vibration_level,oil_pressure,fuel_flow,cabin_pressure,route_type,avg_weather_temp,turbulence,airport_condition,last_maintenance_days,maintenance_score,failure_within_30_days
0,1,AC_103,307,100,863.51,1.23,43.10,6074.29,12.67,Domestic,17.74,0,0.64,123,0.60,0
1,2,AC_180,371,78,765.82,1.86,54.47,6423.24,12.30,Long-haul,5.26,0,0.06,103,0.46,1
2,3,AC_93,214,270,894.46,2.00,60.74,5694.91,11.35,Domestic,-1.77,5,0.66,156,0.64,0
3,4,AC_15,261,183,860.48,1.45,52.37,6198.98,10.16,Domestic,19.30,0,0.69,25,0.91,0
4,5,AC_107,345,81,935.12,1.77,55.67,3752.87,12.65,Domestic,8.79,4,0.52,118,0.51,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,AC_116,494,154,844.79,1.28,51.64,3145.98,12.27,Domestic,22.10,0,0.86,34,0.77,0
49996,49997,AC_33,597,115,823.87,1.73,45.58,6094.27,11.63,International,19.11,4,0.69,167,0.49,0
49997,49998,AC_78,467,235,916.02,2.20,43.47,5025.02,12.33,International,-8.62,0,0.75,75,0.56,0
49998,49999,AC_24,152,195,868.47,1.48,46.80,4169.58,12.35,Long-haul,1.46,0,0.16,143,0.59,0
